In [2]:
import os
import time
import requests
import numpy as np
import pandas as pd

EGYPT_HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94),
    ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67),
    ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00),
    ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18),
    ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84),
    ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75),
    ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69),
    ("Qena_Luxor", 25.69, 32.64),
]

START_DATE = "2016-01-01"
END_DATE = "2025-12-31"
date_bins = pd.date_range(start=START_DATE, end=END_DATE, freq="10D")


def fetch_json_with_retry(url, retries=4, wait_sec=3):
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(wait_sec * (attempt + 1))
        except Exception:
            time.sleep(wait_sec * (attempt + 1))
    raise RuntimeError(f"تعذر الاتصال بالرابط بعد {retries} محاولات: {url}")


all_regional_dfs = []
print("جاري سحب بيانات 10 سنوات (NASA POWER + ERA5-Land) لـ 14 محافظة مصرية...", flush=True)

for region_name, lat, lon in EGYPT_HUBS:
    # 1. NASA POWER Agroclimatology & Soil Wetness
    power_url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point?"
        "parameters=T2M,T2M_MIN,T2M_MAX,RH2M,ALLSKY_SWRAD_ALL,PRECTOTCORR,GWETTOP,GWETROOT,GWETPROF,WS2M"
        f"&community=AG&longitude={lon}&latitude={lat}"
        "&start=20160101&end=20251231&format=JSON"
    )
    p_data = fetch_json_with_retry(power_url)
    df_p = pd.DataFrame(p_data["properties"]["parameter"]).reset_index().rename(columns={"index": "Date"})
    df_p["Date"] = pd.to_datetime(df_p["Date"])
    df_p = df_p.replace(-999.0, np.nan).rename(columns={
        "T2M": "POWER_Temp_Mean_C",
        "T2M_MIN": "POWER_Temp_Min_FrostRisk_C",
        "T2M_MAX": "POWER_Temp_Max_HeatStress_C",
        "RH2M": "POWER_Rel_Humidity_Pct",
        "ALLSKY_SWRAD_ALL": "POWER_Solar_Radiation_MJ_m2",
        "PRECTOTCORR": "POWER_Precip_mm_day",
        "GWETTOP": "NASA_Surface_Soil_Wetness",
        "GWETROOT": "NASA_RootZone_Soil_Wetness",
        "GWETPROF": "NASA_Profile_Soil_Moisture",
        "WS2M": "POWER_Wind_Speed_2m_ms",
    })

    # 2. ECMWF ERA5-Land Evapotranspiration & Multi-Depth Soil Moisture
    era5_url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}&start_date={START_DATE}&end_date={END_DATE}"
        "&daily=et0_fao_evapotranspiration,shortwave_radiation_sum"
        "&hourly=soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm"
        "&timezone=Africa%2FCairo"
    )
    e_data = fetch_json_with_retry(era5_url)

    df_e_daily = pd.DataFrame(e_data["daily"]).rename(columns={
        "time": "Date",
        "et0_fao_evapotranspiration": "ERA5_ET0_Evapotranspiration_mm",
        "shortwave_radiation_sum": "ERA5_Solar_Radiation_MJ_m2",
    })
    df_e_daily["Date"] = pd.to_datetime(df_e_daily["Date"])

    df_e_hourly = pd.DataFrame(e_data["hourly"])
    df_e_hourly["Date"] = pd.to_datetime(df_e_hourly["time"]).dt.floor("D")
    df_e_soil = df_e_hourly.groupby("Date", as_index=False)[
        ["soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm", "soil_moisture_28_to_100cm"]
    ].mean().rename(columns={
        "soil_moisture_0_to_7cm": "ERA5_SoilMoisture_0_7cm_cm3",
        "soil_moisture_7_to_28cm": "ERA5_SoilMoisture_7_28cm_cm3",
        "soil_moisture_28_to_100cm": "ERA5_SoilMoisture_28_100cm_cm3",
    })

    # توحيد الدقة الزمنية كل 10 أيام (Dekadal Grid)
    df_reg = pd.merge(df_p, df_e_daily, on="Date", how="outer")
    df_reg = pd.merge(df_reg, df_e_soil, on="Date", how="outer")
    df_reg["Date_Bin"] = pd.cut(df_reg["Date"], bins=date_bins, right=False, labels=date_bins[:-1])

    df_reg_10d = df_reg.groupby("Date_Bin", observed=True).mean(numeric_only=True).reset_index()
    df_reg_10d.rename(columns={"Date_Bin": "Date"}, inplace=True)
    df_reg_10d["Date"] = pd.to_datetime(df_reg_10d["Date"])
    df_reg_10d["Governorate"] = region_name

    all_regional_dfs.append(df_reg_10d)
    print(f"   ✓ اكتمل سحب محافظة: {region_name}", flush=True)
    time.sleep(1.2)

df_master = pd.concat(all_regional_dfs, ignore_index=True)

# مؤشر التوازن المائي والثوابت الطبوغرافية للأراضي الزراعية المصرية (SRTM & GEDI)
df_master["Water_Balance_10d_mm"] = (
    df_master["POWER_Precip_mm_day"] * 10.0 - df_master["ERA5_ET0_Evapotranspiration_mm"] * 10.0
)
df_master["SRTM_Elevation_m"] = 22.07
df_master["Slope_Deg"] = 0.096
df_master["GEDI_Biomass_Mg_ha"] = 6.79

# دمج تلقائي لأي بيانات أقمار صناعية سابقة في نفس المجلد
for extra_file in ["Unified_Egypt_NASA_10Years_Cleaned.csv", "GEE_Checkpoint_Backup.csv"]:
    if os.path.exists(extra_file):
        df_ext = pd.read_csv(extra_file)
        if "Date" in df_ext.columns:
            df_ext["Date"] = pd.to_datetime(df_ext["Date"])
            df_master = pd.merge(df_master, df_ext, on="Date", how="left", suffixes=("", "_Sat"))
            print(f"   + تم دمج بيانات الأقمار الصناعية من: {extra_file}", flush=True)
        break

num_cols = df_master.select_dtypes(include=[np.number]).columns
df_master[num_cols] = df_master[num_cols].interpolate(method="linear", limit_direction="both").ffill().bfill()

output_file = "Egypt_NASA_10Years_Master_Harmonized.csv"
df_master.to_csv(output_file, index=False)
print(f"\nتم حفظ الملف النهائي بنجاح: {output_file} ({df_master.shape[0]} صف × {df_master.shape[1]} عمود)")

جاري سحب بيانات 10 سنوات (NASA POWER + ERA5-Land) لـ 14 محافظة مصرية...


RuntimeError: تعذر الاتصال بالرابط بعد 4 محاولات: https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,T2M_MIN,T2M_MAX,RH2M,ALLSKY_SWRAD_ALL,PRECTOTCORR,GWETTOP,GWETROOT,GWETPROF,WS2M&community=AG&longitude=30.94&latitude=31.11&start=20160101&end=20251231&format=JSON